# 02 · EDA step 1: choose k for `TimeSeriesSplit`

**Pre-registered rule** (written before running, not changed after; see `docs/eda_plan.md`):
1. **Eligible k:** every test fold and the first training set have at least 150 games.
2. For each eligible k, rank four **fixed** check features by their mean log-loss gain over
   the baseline, across folds. The features are season-to-date goal differential, shot
   share, PP% and save %.
3. Choose the **smallest eligible k whose ranking agrees with every larger eligible k**
   (Spearman ≥ 0.8).
4. If none qualifies, use **k = 5**.

This notebook prints **stability numbers only**. It never shows feature names next to their
scores, so this step can't steer which features get favored later.

In [ ]:
%run ./01_pregame_features.ipynb

In [ ]:
import warnings

import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CHECK_FEATURES = [
    "diff_goal_diff_pg_season",
    "diff_shot_share_season",
    "diff_pp_pct_season",
    "diff_save_pct_season",
]
K_RANGE = range(3, 9)
MIN_FOLD = 150
AGREEMENT = 0.8
DEFAULT_K = 5

## Helpers

In [ ]:
def fold_sizes(n: int, k: int) -> tuple[int, int]:
    """(first training set, smallest test fold) for TimeSeriesSplit(k) on n rows."""
    splits = list(TimeSeriesSplit(n_splits=k).split(np.zeros(n)))
    return len(splits[0][0]), min(len(test) for _, test in splits)


def eligible(n: int, ks=K_RANGE, min_fold: int = MIN_FOLD) -> list[int]:
    return [k for k in ks if min(fold_sizes(n, k)) >= min_fold]


def fold_scores(df: pd.DataFrame, k: int, features: list[str]) -> pd.DataFrame:
    """Log-loss gain over the baseline per fold (rows) and feature (columns).

    The baseline always predicts the training win rate. Positive = the feature beat it.
    """
    y = df["home_win"].to_numpy()
    rows = []
    for train, test in TimeSeriesSplit(n_splits=k).split(df):
        base = log_loss(y[test], np.full(len(test), y[train].mean()), labels=[0, 1])
        row = {}
        for f in features:
            x = df[[f]].to_numpy()
            model = make_pipeline(StandardScaler(), LogisticRegression()).fit(x[train], y[train])
            row[f] = base - log_loss(y[test], model.predict_proba(x[test])[:, 1], labels=[0, 1])
        rows.append(row)
    return pd.DataFrame(rows)


def agreement(r1: pd.Series, r2: pd.Series) -> float:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rho = spearmanr(r1.sort_index(), r2.sort_index()).statistic
    return 1.0 if np.isnan(rho) else float(rho)


def choose(rankings: dict, threshold: float = AGREEMENT, default=DEFAULT_K):
    ks = sorted(rankings)
    for i, k in enumerate(ks):
        later = ks[i + 1:]
        if later and all(agreement(rankings[k], rankings[j]) >= threshold for j in later):
            return k
    return default

### Helper checks

In [ ]:
assert eligible(1000) == [3, 4, 5]  # k=6 gives test folds of 142

_r = lambda *ranks: pd.Series(ranks, index=list("abcd"), dtype=float)
assert choose({3: _r(3, 1, 2, 4), 4: _r(1, 2, 3, 4), 5: _r(1, 2, 3, 4)}) == 4  # k=3 agrees at 0.4
assert choose({3: _r(1, 2, 3, 4), 4: _r(4, 3, 2, 1), 5: _r(1, 2, 3, 4)}) == DEFAULT_K
print("Helper checks passed.")

## Run the check

- **spread:** typical fold-to-fold SD of a feature's gain.
- **gap:** typical gap between neighbouring features in the ranking.
- **gap/spread above 1** means feature differences are bigger than fold noise.

In [ ]:
df = rows.dropna(subset=CHECK_FEATURES).reset_index(drop=True)
print(f"n = {len(df)} games ({len(rows) - len(df)} dropped for missing check features)\n")

ok = eligible(len(df))
rankings, summary = {}, []
for k in K_RANGE:
    first, smallest = fold_sizes(len(df), k)
    scores = fold_scores(df, k, CHECK_FEATURES)
    means = scores.mean().sort_values(ascending=False)
    spread = scores.std(ddof=1).mean()
    gap = means.diff().abs().mean()
    if k in ok:
        rankings[k] = scores.mean().rank(ascending=False)
    summary.append({
        "k": k, "first train": first, "min test": smallest, "eligible": k in ok,
        "spread": round(spread, 4), "gap": round(gap, 4), "gap/spread": round(gap / spread, 2),
        "beat baseline": f"{(scores > 0).to_numpy().mean():.0%}",
    })

pd.DataFrame(summary).set_index("k")

In [ ]:
ks = sorted(rankings)
pd.DataFrame(
    [[round(agreement(rankings[a], rankings[b]), 2) for b in ks] for a in ks],
    index=pd.Index(ks, name="k"), columns=ks,
)

In [ ]:
k_rule = choose(rankings, default=None)
if k_rule is None:
    print(f"No k met the stability rule -> default k = {DEFAULT_K}")
else:
    print(f"Rule picks k = {k_rule} (smallest eligible k whose ranking matches every larger one)")

## Decision: lock k = 5

**The rule picks k = 3, and we lock k = 5 instead.** The reasons:

1. **Stability can't tell 3, 4 and 5 apart.** The feature ranking is identical at every
   eligible k (all agreements are 1.0). The rule picks the smallest k only as a tie-break,
   so changing that tie-break uses no information about which features score better. It's
   not tuning on the validation data.
2. **The Step 5 safeguards need folds.** Features are chosen on folds 1 to k−1, and the
   shortlist is confirmed once on the last fold. A feature must beat the baseline in most
   folds.
   - At k = 3: 2 folds to choose from, and "most" means 2 out of 2.
   - At k = 5: 4 folds to choose from, and a real consistency test.
3. **k = 5 still meets the size rule.** The smallest test fold has 159 games and the first
   training set has 162 (both ≥ 150).
4. **Larger k isn't possible:** at k = 6 the test folds drop below 150 games.

**The fix for the rule, from here on:** among eligible k values whose rankings agree
equally, take the largest. That leaves the most folds for choosing features and confirming
them.

The cell below locks k = 5 only if conditions 1 and 3 hold. If the data changes and they
fail, it falls back to the rule's pick.

In [ ]:
K_LOCKED = 5

identical = all(agreement(rankings[a], rankings[b]) == 1.0 for a in ks for b in ks)
k5_ok = K_LOCKED in ok

if identical and k5_ok:
    K = K_LOCKED
    first, smallest = fold_sizes(len(df), K)
    print(f"k = {K} locked. Rankings identical at k = {ks}, so this override uses no score information.")
    print(f"At k = {K}: first training set {first} games, smallest test fold {smallest} games.")
    print(f"Step 5 chooses features on folds 1 to {K - 1} and confirms once on fold {K}.")
else:
    K = k_rule if k_rule is not None else DEFAULT_K
    print(f"Override conditions not met (identical={identical}, k=5 eligible={k5_ok}); using k = {K}.")